# Lab: The view from the driver's seat

You will write the three functions the lecture ends with: `rotation_matrix`, `world_to_ego` and `ego_to_world`. You will check
`world_to_ego` against the simulator's own transform at 40 places and headings, put it into the camera so that every
picture your car sees passes through your code, and find out why a test at yaw 0 proves nothing.
Read the lecture first; this lab points back to its sections by title and follows its order.

Run the cells from top to bottom (Shift+Enter). Each problem has a "Guess first" line: write your guess down before you
run the cell, then compare. `practice` prints a green tick or tells you what it expected. A "Break it" cell makes a
mistake on purpose, so a red cross there is what it is meant to show.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/result_first.png" alt="The provided car 150 m along gentle, drawn twice in two stacked panels. Top: seen from above in the world frame, east to the right and north up. The car&#x27;s front axle is at (147.839, 14.299), and its own forward and left axes, x and y, are drawn in purple, turned 0.5958 rad from east. Bottom: the same lane lines in the car&#x27;s own frame, with forward up the page and the car&#x27;s left to the left. The lane centre starts 0.527 m to the car&#x27;s right. The lane-centre points 10, 20 and 30 m further along the lane are 0.453, 2.694 and 6.160 m to its left.">
<figcaption><em>Lecture figure 1. The provided car 150 m along gentle, drawn twice in two stacked panels. Top: seen from above in the world frame, east to the right and north up. The car&#x27;s front axle is at (147.839, 14.299), and its own forward and left axes, x and y, are drawn in purple, turned 0.5958 rad from east. Bottom: the same lane lines in the car&#x27;s own frame, with forward up the page and the car&#x27;s left to the left. The lane centre starts 0.527 m to the car&#x27;s right. The lane-centre points 10, 20 and 30 m further along the lane are 0.453, 2.694 and 6.160 m to its left.</em></figcaption>
</figure>

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install -q "git+https://github.com/vvknyn/self-driving-ai-course@main"
%matplotlib inline

In [ ]:
import math

import numpy as np

from zero2fsd.car import Car, run
from zero2fsd.geom import forward_left, heading_vector, wrap_angle
from zero2fsd.grade import check, practice
from zero2fsd.preview.units.u1_1_2 import PICTURES, differing_pictures
from zero2fsd.sim import BicycleState, Camera, make_scenario
from zero2fsd.sim.camera import WHITE_LINE_LATERAL
from zero2fsd.sim.world import EGO_LANE_CENTER

## The scene

The lecture's scene is the provided car 150 m along `gentle`, at step 378 of its lap. The next cell finds that moment and
the lane-centre points 10, 20 and 30 m further along the lane. `SCENE_POSE` is the car's pose `(x, y, theta)`;
`lane_points` has one row per point, east part first.

In [ ]:
road = make_scenario("gentle").road
lap = run(Car(), "gentle", max_steps=379)
SCENE_POSE = (float(lap.x[378]), float(lap.y[378]), float(lap.yaw[378]))
x, y, theta = SCENE_POSE
t = np.array([x, y])
s_car, _ = road.project(x, y)
lane_points = np.array([road.offset(s_car + ahead, EGO_LANE_CENTER) for ahead in (10.0, 20.0, 30.0)])
p = lane_points[0]
print(f"car at ({x:.3f}, {y:.3f}), theta {theta:.4f} rad; first lane-centre point at ({p[0]:.3f}, {p[1]:.3f})")

## Two dot products make a frame change

The arrow from the car to the first lane-centre point is $\mathbf{v} = \mathbf{p} - \mathbf{t}$. *Positions and headings*
turned an arrow like it into "ahead and to the left" with `forward_left`.

**Problem 1.** Compute `v`, then `forward_left(theta, v)`, and unpack the pair into `forward` and `left`.

**Guess first:** how far ahead of the car is the point, and how far to its left? Compare with the lecture's 10.189 m and
0.453 m.

In [ ]:
v = None  # your code: the arrow from the car to the point
forward = left = None  # your code: the two parts of v, from forward_left

practice("forward (m)", forward, 10.189, tol=0.001)
practice("left (m)", left, 0.453, tol=0.001)

**Break it.** Take the dot product with the left arrow first and the heading arrow second, and the pair comes out swapped.
The cell makes that mistake; the red cross is what you should see.

In [ ]:
h = heading_vector(theta)
n = np.array([-math.sin(theta), math.cos(theta)])
swapped = (float(v @ n), float(v @ h)) if v is not None else None
practice("forward, left with the arrows in the wrong order", swapped, (10.189, 0.453), tol=0.001)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/two_dots.png" alt="The arrow v from the car&#x27;s front axle to the lane-centre point 10 m further along, 8.179 m east and 6.093 m north, with the car&#x27;s heading arrow and left arrow drawn in purple. Dashed lines from the tip of v meet the two arrows&#x27; directions at right angles: 10.189 m along the heading arrow, and 0.453 m along the left arrow.">
<figcaption><em>Lecture figure 2. The arrow v from the car&#x27;s front axle to the lane-centre point 10 m further along, 8.179 m east and 6.093 m north, with the car&#x27;s heading arrow and left arrow drawn in purple. Dashed lines from the tip of v meet the two arrows&#x27; directions at right angles: 10.189 m along the heading arrow, and 0.453 m along the left arrow.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/stacked_dots.png" alt="The matrix–vector product at the scene, drawn as two dot products stacked. Top row: the heading arrow, (0.8277, 0.5612), dotted with v = (8.179, 6.093), gives the forward part, 10.189. Bottom row: the left arrow, (−0.5612, 0.8277), dotted with v gives the left part, 0.453.">
<figcaption><em>Lecture figure 3. The matrix–vector product at the scene, drawn as two dot products stacked. Top row: the heading arrow, (0.8277, 0.5612), dotted with v = (8.179, 6.093), gives the forward part, 10.189. Bottom row: the left arrow, (−0.5612, 0.8277), dotted with v gives the left part, 0.453.</em></figcaption>
</figure>

## The rotation matrix

**Problem 2.** Fill in `rotation_matrix(theta)`: the (2, 2) array whose first column is the heading arrow
$(\cos\theta, \sin\theta)$ and whose second column is the left arrow $(-\sin\theta, \cos\theta)$.

**Guess first:** what is `rotation_matrix(0)`?

In [ ]:
def rotation_matrix(theta):
    """The (2, 2) matrix whose columns are the heading arrow and the left arrow.

    Input:   theta, the yaw in radians.
    Output:  a (2, 2) array.
    Example: rotation_matrix(0)  ->  [[1, 0], [0, 1]]
    """
    raise NotImplementedError


R = rotation_matrix(theta)
left_arrow = np.array([-math.sin(theta), math.cos(theta)])
practice("column 1 minus the heading arrow (largest entry)", float(np.abs(R[:, 0] - heading_vector(theta)).max()), 0, tol=1e-12)
practice("column 2 minus the left arrow (largest entry)", float(np.abs(R[:, 1] - left_arrow).max()), 0, tol=1e-12)

**Break it.** Write the sine in the wrong places, `[[c, s], [-s, c]]`. The columns stop being the two arrows.

In [ ]:
c, s = math.cos(theta), math.sin(theta)
wrong = np.array([[c, s], [-s, c]])
practice("column 1 minus the heading arrow, wrong matrix", float(np.abs(wrong[:, 0] - heading_vector(theta)).max()), 0, tol=1e-12)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/columns_axes.png" alt="Two stacked panels at the scene. Top: the two columns of R(θ) at θ = 0.5958 rad, (0.8277, 0.5612) and (−0.5612, 0.8277), drawn in purple from the car&#x27;s front axle. They are the ego frame&#x27;s x and y axes. The world&#x27;s x and y axes are drawn in grey for comparison. Bottom: start at the car, (147.839, 14.299), walk 10.189 m along the first column and then 0.453 m along the second, and you arrive at the lane-centre point (156.018, 20.391).">
<figcaption><em>Lecture figure 4. Two stacked panels at the scene. Top: the two columns of R(θ) at θ = 0.5958 rad, (0.8277, 0.5612) and (−0.5612, 0.8277), drawn in purple from the car&#x27;s front axle. They are the ego frame&#x27;s x and y axes. The world&#x27;s x and y axes are drawn in grey for comparison. Bottom: start at the car, (147.839, 14.299), walk 10.189 m along the first column and then 0.453 m along the second, and you arrive at the lane-centre point (156.018, 20.391).</em></figcaption>
</figure>

## Undo is the transpose

**Problem 3.** Write `world_to_ego(pose, p_world)` and `ego_to_world(pose, p_ego)`. The lecture's formulas are
$R(\theta)^{\mathsf T}(\mathbf{p} - \mathbf{t})$ and $R(\theta)\,\mathbf{q} + \mathbf{t}$. Use `rotation_matrix` and write
the subtraction before the rotation. Keep the arrays as rows for now: one point of shape `(2,)` is enough, and the next
section makes it work on every point at once.

**Guess first:** what does `world_to_ego` give for the car's own position?

The first function below is the one Exercise A and Exercise B check, so its cell carries the exercise tags.

In [ ]:
def world_to_ego(pose, p_world):
    """World point(s) in the car's own frame: how far ahead of the car, and how far to its left.

    Input:   pose = (x, y, theta), the car's position and yaw; p_world, an array of shape (..., 2), one point per row,
             east part first.
    Output:  an array of the same shape (..., 2): forward, then left, in metres.
    Example: world_to_ego(SCENE_POSE, [156.018, 20.391])  ->  [10.189..., 0.453...]
    """
    raise NotImplementedError

In [ ]:
def ego_to_world(pose, p_ego):
    """Points given in the car's own frame (forward, left) back in the world: R(theta) q + t, one point per row."""
    raise NotImplementedError


practice("scene point, forward then left", world_to_ego(SCENE_POSE, p), (10.189, 0.453), tol=0.001)
practice("largest entry of R transposed times R, minus the identity", float(np.abs(R.T @ R - np.eye(2)).max()), 0, tol=1e-12)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/undo.png" alt="The same arrow turned out and back, in two stacked panels, with the car&#x27;s front axle at the origin and the world&#x27;s x and y axes in grey. Top: R(θ) turns the arrow (10.189, 0.453) by 0.5958 rad anticlockwise, to (8.179, 6.093). Bottom: R(θ)ᵀ turns (8.179, 6.093) by the same angle clockwise, back to (10.189, 0.453).">
<figcaption><em>Lecture figure 5. The same arrow turned out and back, in two stacked panels, with the car&#x27;s front axle at the origin and the world&#x27;s x and y axes in grey. Top: R(θ) turns the arrow (10.189, 0.453) by 0.5958 rad anticlockwise, to (8.179, 6.093). Bottom: R(θ)ᵀ turns (8.179, 6.093) by the same angle clockwise, back to (10.189, 0.453).</em></figcaption>
</figure>

**Break it.** Rotate first and subtract second: $R(\theta)^{\mathsf T}\mathbf{p} - \mathbf{t}$. The cell does it for the
same point. The lecture's answer is (-7.260, -84.974).

In [ ]:
rotate_first = R.T @ p - t
practice("rotate first, then subtract t", tuple(float(c) for c in rotate_first), (10.189, 0.453), tol=0.001)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/order_trap.png" alt="Two answers for the lane-centre point 10 m further along, in the car&#x27;s own frame with forward up the page and the car at the origin. Subtract the car&#x27;s position first, then turn: (10.189, 0.453), just ahead of the car and a little to its left. Turn first, then subtract: (−7.260, −84.974), 7.260 m behind the car and 84.974 m to its right, nowhere near the road.">
<figcaption><em>Lecture figure 6. Two answers for the lane-centre point 10 m further along, in the car&#x27;s own frame with forward up the page and the car at the origin. Subtract the car&#x27;s position first, then turn: (10.189, 0.453), just ahead of the car and a little to its left. Turn first, then subtract: (−7.260, −84.974), 7.260 m behind the car and 84.974 m to its right, nowhere near the road.</em></figcaption>
</figure>

Now the lap. The next cell sends the 2109 road points of `gentle` into the car's frame and back out. Neither number is
exactly 0, because every multiply-and-add rounds in the last digit.

In [ ]:
cloud = np.concatenate([road.sample_points(1.0, d) for d in (0.0, EGO_LANE_CENTER, WHITE_LINE_LATERAL)])
round_trip = float(np.abs(ego_to_world(SCENE_POSE, world_to_ego(SCENE_POSE, cloud)) - cloud).max())
print(f"{len(cloud)} points")
practice("largest round-trip error (m)", round_trip, 0, tol=1e-12)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/round_trip.png" alt="Two stacked panels. Top: the lap of gentle in the world frame, its 2109 points in grey, and the same points after world_to_ego and then ego_to_world at the scene&#x27;s pose, in purple, landing on top of them. Bottom: at each of the 41 poses, the largest coordinate error after the round trip, on a scale where each grid line is ten times the one below. Every pose is below the dashed line at 10 to the minus 12 m. The largest error is 1.4 times ten to the minus 13 m going world to ego to world, and 8.5 times ten to the minus 14 m going ego to world to ego.">
<figcaption><em>Lecture figure 7. Two stacked panels. Top: the lap of gentle in the world frame, its 2109 points in grey, and the same points after world_to_ego and then ego_to_world at the scene&#x27;s pose, in purple, landing on top of them. Bottom: at each of the 41 poses, the largest coordinate error after the round trip, on a scale where each grid line is ten times the one below. Every pose is below the dashed line at 10 to the minus 12 m. The largest error is 1.4 times ten to the minus 13 m going world to ego to world, and 8.5 times ten to the minus 14 m going ego to world to ego.</em></figcaption>
</figure>

## Every point at once

The camera calls `world_to_ego` three times per picture, each time with a whole array: 146 road quads of 4 corners, the
same for the white line, and 103 yellow dashes.

**Problem 4.** Check that your `world_to_ego` returns the same shape it was given, for the road's array of shape
`(146, 4, 2)`, and that its answers for the three lane-centre points match the lecture's table.

**Guess first:** does `rot @ rows` work with 2 points? With 5?

In [ ]:
quads = np.zeros((146, 4, 2)) + lane_points[0]
practice("output shape for the road quads", world_to_ego(SCENE_POSE, quads).shape, (146, 4, 2))

table = world_to_ego(SCENE_POSE, lane_points)
practice("the three lane-centre points: forward, then left", table, [[10.189, 0.453], [20.158, 2.694], [29.771, 6.160]], tol=0.001)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/quads.png" alt="The road around the car parked on the lane centre 150 m along gentle, seen from above, with every quad outlined: strips of road surface 0.5 m long, strips of white line, and yellow dashes. One road quad is shaded, with its 4 corners marked as dots. Each corner is 2 numbers, east and north.">
<figcaption><em>Lecture figure 8. The road around the car parked on the lane centre 150 m along gentle, seen from above, with every quad outlined: strips of road surface 0.5 m long, strips of white line, and yellow dashes. One road quad is shaded, with its 4 corners marked as dots. Each corner is 2 numbers, east and north.</em></figcaption>
</figure>

**Break it.** The maths wrote arrows as columns, so `rot @ points` looks right. With rows it is not.

In [ ]:
rot = np.array([[c, -s], [s, c]])
try:
    rot @ lane_points[:2].copy()
    print("2 points: no error, and the answer is wrong")
except ValueError as err:
    print("2 points:", err)
try:
    rot @ np.zeros((5, 2))
except ValueError as err:
    print("5 points: ValueError, the shapes do not line up")
print("rows @ rot works for both:", (lane_points[:2] - t) @ rot)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/shape_trap.png" alt="Two lane-centre points, 160 m and 170 m along gentle, in the ego frame of the car 150 m along the road, with forward up the page. Purple dots: the rows times R(θ), the right product, at (9.837, 0.398) and (19.817, 2.595), on the lane centre ahead of the car. Grey crosses: R(θ) times the rows, the same two arrays the other way round, read as two points: (−1.833, −2.604) and (16.815, 14.265), off the road, with no error message.">
<figcaption><em>Lecture figure 9. Two lane-centre points, 160 m and 170 m along gentle, in the ego frame of the car 150 m along the road, with forward up the page. Purple dots: the rows times R(θ), the right product, at (9.837, 0.398) and (19.817, 2.595), on the lane centre ahead of the car. Grey crosses: R(θ) times the rows, the same two arrays the other way round, read as two points: (−1.833, −2.604) and (16.815, 14.265), off the road, with no error message.</em></figcaption>
</figure>

**Exercise A.** The check puts a point 4 m ahead and 1 m left of a car at yaws 0, π/2, π and −3π/4, where it must come out
as (4, 1); then tries the `(K, 4, 2)` shape; then the round trip; then 40 seeded poses, with yaws up to ±4π, against the
simulator's own transform. Expect a largest difference of about 5.7e-14 m for the matrix form
`(p_world - t) @ rot`; the check allows 1e-9 m.

In [ ]:
check("1.1.2.a", world_to_ego)

## Swap your transform into the renderer

The camera draws with `geom.world_to_ego` unless you give it another: `Camera(world_to_ego=your_function)`.

**Problem 5.** Take 18 evenly spaced moments of the provided car's lap of `gentle`, draw each through the simulator's
transform and through yours, and count the pictures that differ.

**Guess first:** how many of the 18 pictures will differ, and by how many pixels?

In [ ]:
lap_full = run(Car(), "gentle")
pictures, pixels = differing_pictures(world_to_ego, lap_full)
print(f"{pictures} of {PICTURES} pictures differ, {pixels} pixels")
practice("pictures that differ", pictures, 0, tol=0)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/renderer_steps.png" alt="How the camera draws one picture, in three stacked panels, for the car parked on the lane centre 150 m along gentle and pointing along the road, at yaw 0.6254 rad. Top: the road&#x27;s quads in the world frame, from above. Middle: the same quads after world_to_ego, in the ego frame with forward up the page, from 3 m behind the car to 70 m ahead. Bottom: the 320 by 180 pixel picture painted from the middle panel&#x27;s quads.">
<figcaption><em>Lecture figure 10. How the camera draws one picture, in three stacked panels, for the car parked on the lane centre 150 m along gentle and pointing along the road, at yaw 0.6254 rad. Top: the road&#x27;s quads in the world frame, from above. Middle: the same quads after world_to_ego, in the ego frame with forward up the page, from 3 m behind the car to 70 m ahead. Bottom: the 320 by 180 pixel picture painted from the middle panel&#x27;s quads.</em></figcaption>
</figure>

**Exercise B** puts your function in the camera and drives `gentle`. The provided car must complete it with a mean
distance from the lane centre under 0.5 m. With the right transform the lap is identical to the provided one, near 0.376 m.

In [ ]:
check("1.1.2.b", world_to_ego)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/swap_same.png" alt="Two camera pictures of the same moment, stacked: the provided car 151.1 m along gentle, 19.00 s into its lap. Top: drawn by the simulator&#x27;s own transform. Bottom: drawn through a world_to_ego written as rows times R(θ). Not one of the 57600 pixels differs.">
<figcaption><em>Lecture figure 11. Two camera pictures of the same moment, stacked: the provided car 151.1 m along gentle, 19.00 s into its lap. Top: drawn by the simulator&#x27;s own transform. Bottom: drawn through a world_to_ego written as rows times R(θ). Not one of the 57600 pixels differs.</em></figcaption>
</figure>

## A test at yaw 0 proves nothing

Two broken transforms: the swap, `(p - t) @ rot.T` (R where its transpose belongs), and no rotation, `p - t`. Each is
right when the car points along east.

**Problem 6.** Run both on the lecture's named poses, then count how much of the parked car's picture each changes at
three places on the lap.

**Guess first:** at which yaws will each one give (4, 1)?

In [ ]:
def swap(pose, p_world):
    x, y, theta = pose
    return (np.asarray(p_world, dtype=float) - np.array([x, y])) @ np.array([[math.cos(theta), -math.sin(theta)], [math.sin(theta), math.cos(theta)]]).T


def no_rotation(pose, p_world):
    x, y, theta = pose
    return np.asarray(p_world, dtype=float) - np.array([x, y])


for label, yaw in (("0", 0.0), ("pi/2", math.pi / 2), ("pi", math.pi), ("-3pi/4", -3 * math.pi / 4)):
    point = np.array([10.0, 5.0]) + 4.0 * heading_vector(yaw) + 1.0 * np.array([-math.sin(yaw), math.cos(yaw)])
    print(f"yaw {label:>7}: swap {np.round(swap((10.0, 5.0, yaw), point), 3)}, no rotation {np.round(no_rotation((10.0, 5.0, yaw), point), 3)}")

Parked on the lane centre, the car sees a different picture through a broken transform. `parked_share` is the percent of
the picture's pixels, by what they truly are, that the broken transform changes.

In [ ]:
def parked_share(fn, s):
    state = BicycleState(*road.ego_pose(s), 8.0, 0.0)
    good = Camera().render(road, state, return_labels=True)[1]
    bad = Camera(world_to_ego=fn).render(road, state, return_labels=True)[1]
    return 100.0 * float((good != bad).mean())


practice("swap at the start line (%)", parked_share(swap, 0.0), 0.0, tol=0.05)
practice("swap 150 m along (%)", parked_share(swap, 150.0), 33.9, tol=0.05)
practice("no rotation 150 m along (%)", parked_share(no_rotation, 150.0), 20.0, tol=0.05)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/pictures_150.png" alt="Three camera pictures of the same moment, stacked: the car parked on the lane centre 150 m along gentle, at yaw 0.6254 rad. Top: through the right transform. Middle: through the swap, which changes what 33.9% of the pixels show. Bottom: with no rotation, which changes 20.0%.">
<figcaption><em>Lecture figure 12. Three camera pictures of the same moment, stacked: the car parked on the lane centre 150 m along gentle, at yaw 0.6254 rad. Top: through the right transform. Middle: through the swap, which changes what 33.9% of the pixels show. Bottom: with no rotation, which changes 20.0%.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/share_vs_s.png" alt="Two stacked panels against distance along gentle, from 0 m to 400 m, for a car parked on the lane centre at seven places. Top: the car&#x27;s yaw, in orange. Bottom: the share of pixels each mistake changes, the swap solid and no rotation dashed. Both are 0.0% at 0 m and 50 m, where the yaw is 0. At 400 m, on the back straight, the yaw is π = 3.1416 rad and the swap is back to 0.0%, while no rotation changes 26.2%.">
<figcaption><em>Lecture figure 13. Two stacked panels against distance along gentle, from 0 m to 400 m, for a car parked on the lane centre at seven places. Top: the car&#x27;s yaw, in orange. Bottom: the share of pixels each mistake changes, the swap solid and no rotation dashed. Both are 0.0% at 0 m and 50 m, where the yaw is 0. At 400 m, on the back straight, the yaw is π = 3.1416 rad and the swap is back to 0.0%, while no rotation changes 26.2%.</em></figcaption>
</figure>

The swap changes nothing at the start line, where the car points along east, yet it leaves `straight` after about three
seconds. The next cell drives it.

In [ ]:
swap_run = run(Car(), "straight", camera=Camera(world_to_ego=swap))
print(f"completed: {swap_run.completed}; left the road after {swap_run.t[-1]:.2f} s, {swap_run.progress_m[-1]:.1f} m along")
practice("seconds before the swap leaves straight", float(swap_run.t[-1]), 2.95, tol=0.05)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/e_vs_s.png" alt="Distance from the lane centre, e, against distance along the road, for the provided car seeing through three transforms, in two stacked panels. Top, straight: through the right transform e stays within 0.002 m, and with no rotation within 0.003 m; through the swap the car crosses the yellow line 22.8 m from the start, 2.95 s in. Bottom, gentle: the right transform finishes the lap; the swap leaves the lane at 21.7 m, and no rotation at 116.6 m, 16.6 m into the first bend.">
<figcaption><em>Lecture figure 14. Distance from the lane centre, e, against distance along the road, for the provided car seeing through three transforms, in two stacked panels. Top, straight: through the right transform e stays within 0.002 m, and with no rotation within 0.003 m; through the swap the car crosses the yellow line 22.8 m from the start, 2.95 s in. Bottom, gentle: the right transform finishes the lap; the swap leaves the lane at 21.7 m, and no rotation at 116.6 m, 16.6 m into the first bend.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/heading_flip.png" alt="What the perception box reads as the heading, against the car&#x27;s true heading error ψ, at the 53 steps of the swapped car&#x27;s run on straight where ψ is more than 0.001 rad from 0. Red dots, read through the swap: the box&#x27;s heading is the true ψ with its sign flipped, a median ratio of −0.994. Grey dots, the same moments read through the right transform, 49 of them with a valid reading: a median ratio of +1.011.">
<figcaption><em>Lecture figure 15. What the perception box reads as the heading, against the car&#x27;s true heading error ψ, at the 53 steps of the swapped car&#x27;s run on straight where ψ is more than 0.001 rad from 0. Red dots, read through the swap: the box&#x27;s heading is the true ψ with its sign flipped, a median ratio of −0.994. Grey dots, the same moments read through the right transform, 49 of them with a valid reading: a median ratio of +1.011.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/park_box.png" alt="Two camera pictures of a car parked on straight, stacked: 50 m from the start, 0.3 m left of the lane centre, pointing 0.05 rad right. Top: through the right transform; the perception box reads an offset of 0.3063 m and a heading of −0.0517 rad. Bottom: through the swap; it reads 0.2964 m and +0.0506 rad.">
<figcaption><em>Lecture figure 16. Two camera pictures of a car parked on straight, stacked: 50 m from the start, 0.3 m left of the lane centre, pointing 0.05 rad right. Top: through the right transform; the perception box reads an offset of 0.3063 m and a heading of −0.0517 rad. Bottom: through the swap; it reads 0.2964 m and +0.0506 rad.</em></figcaption>
</figure>

**Fix it.** The way to catch both mistakes is the one Exercise A already uses: yaws where sin θ is well away from 0, and
many random poses. Your `world_to_ego` passed it, and the picture test, so the camera now draws through your code.
The lecture's last figure shows what the next lesson is about.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/next_lesson.png" alt="The car 150 m along gentle, seen from above, 0.527 m to the left of its lane centre. Two braces carry a question mark each: one follows the road from its start to the car, and one crosses the lane from the lane centre to the car&#x27;s front axle.">
<figcaption><em>Lecture figure 17. The car 150 m along gentle, seen from above, 0.527 m to the left of its lane centre. Two braces carry a question mark each: one follows the road from its start to the car, and one crosses the lane from the lane centre to the car&#x27;s front axle.</em></figcaption>
</figure>

The lecture's two challenges end with these figures: the swap seen through time, and how the error grows.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/challenge_swing.png" alt="The provided car on straight seeing through the swap, from the start to 2.50 s, in two stacked panels against time. Top: distance from the lane centre, e. Bottom: heading error, ψ. e is most negative, −0.2824 m, at 2.10 s; ψ crosses zero at 2.30 s; and e is back through zero at 2.49 s.">
<figcaption><em>Lecture figure 18. The provided car on straight seeing through the swap, from the start to 2.50 s, in two stacked panels against time. Top: distance from the lane centre, e. Bottom: heading error, ψ. e is most negative, −0.2824 m, at 2.10 s; ψ crosses zero at 2.30 s; and e is back through zero at 2.49 s.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.2-the-view-from-the-drivers-seat/figures/challenge_growth.png" alt="Distance from the lane centre e against time on straight, for the provided car seeing through the swap, from the start to 2.50 s. A dashed line shows the step-by-step model started from the measured offset and heading at 0.40 s. It predicts −0.0456, −0.1515 and −0.3061 m at 1.00, 1.50 and 2.00 s, against −0.0382, −0.1339 and −0.2724 m measured. With the right transform, e stays within 0.002 m.">
<figcaption><em>Lecture figure 19. Distance from the lane centre e against time on straight, for the provided car seeing through the swap, from the start to 2.50 s. A dashed line shows the step-by-step model started from the measured offset and heading at 0.40 s. It predicts −0.0456, −0.1515 and −0.3061 m at 1.00, 1.50 and 2.00 s, against −0.0382, −0.1339 and −0.2724 m measured. With the right transform, e stays within 0.002 m.</em></figcaption>
</figure>